# Day H1 — 진단, 예후, 조기진단: 문제를 먼저 정의하기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/H-clinical-prediction/day-H1.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/H-clinical-prediction/day-H1.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 같은 도구, 세 가지 유병률

민감도 85%, 특이도 90%인 가상의 선별 도구를 1만 명에게 쓴다고 합시다.

In [ ]:
import numpy as np
import pandas as pd

sens, spec = 0.85, 0.90          # 가상의 선별 도구: 민감도 85%, 특이도 90%
rows = []
for setting, prev in [("환자-대조군 연구(1:1)", 0.50),
                      ("고위험 의뢰 클리닉", 0.10),
                      ("일반 인구 선별", 0.005)]:
    n = 10000
    diseased = n * prev
    tp = sens * diseased
    fp = (1 - spec) * (n - diseased)
    fn = diseased - tp
    tn = (n - diseased) - fp
    rows.append({"상황": setting, "유병률": prev, "양성 판정": round(tp + fp),
                 "진양성": round(tp), "위양성": round(fp),
                 "PPV": round(tp / (tp + fp), 3), "NPV": round(tn / (tn + fn), 4)})
print(pd.DataFrame(rows).to_string(index=False))

### 예시 2 — 스펙트럼 편향을 만들어 보기

질환 진행 정도(0–1)에 따라 오르는 가상 지표 하나로, 누구와 누구를 비교하느냐에 따라 AUC가 어떻게 달라지는지 봅니다.

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(1)
# 가상의 혈액·센서 지표 하나. 질환이 진행할수록(severity 0→1) 지표가 오른다.
def marker(severity, n):
    return 2.0 * severity + rng.normal(0, 1, n)

n = 400
established = marker(rng.uniform(0.7, 1.0, n), n)   # 진단이 확정된 중등도 이상 환자
prodromal   = marker(rng.uniform(0.0, 0.4, n), n)   # 아직 진단 전, 전구기
healthy     = marker(np.zeros(n), n)                # 건강한 자원자
mimics      = marker(np.full(n, 0.5), n)            # 비슷한 증상을 보이는 다른 질환

def auc(pos, neg):
    y = np.r_[np.ones(len(pos)), np.zeros(len(neg))]
    return roc_auc_score(y, np.r_[pos, neg])

print(f"확정 환자 vs 건강 자원자       AUC {auc(established, healthy):.3f}")
print(f"확정 환자 vs 감별 대상 질환     AUC {auc(established, mimics):.3f}")
print(f"전구기 vs 건강 자원자           AUC {auc(prodromal, healthy):.3f}")
print(f"전구기 vs (건강 + 감별 대상)    AUC {auc(prodromal, np.r_[healthy, mimics]):.3f}")